# HDF5 to CSV Converter

Convert experiment HDF5 files to CSV for external analysis tools.

In [ ]:
import sys, os
sys.path.insert(0, r"C:\Networking Experiment\Claude_TEST_faisal\useful_codes\repository")

import numpy as np
import h5py
import csv
import glob
import datetime
from pathlib import Path

from analysis.data_loader import load_h5, find_runs, RESULTS_DIR
print("Results directory:", RESULTS_DIR)
print("Ready.")

## 1. Select File

In [ ]:
########################################
RID = None   # Set RID, or None for latest
########################################

if RID is not None:
    pattern = "%09d-*.h5" % RID
    matches = glob.glob(str(RESULTS_DIR / "**" / pattern), recursive=True)
    if not matches:
        raise FileNotFoundError("No HDF5 for RID %d" % RID)
    filepath = max(matches, key=os.path.getmtime)
else:
    all_files = find_runs(format="h5")
    if not all_files:
        raise FileNotFoundError("No HDF5 files")
    filepath = all_files[-1]

print("File:", os.path.basename(filepath))
print("Path:", filepath)

result = load_h5(filepath)
data = result["data"]
meta = result["metadata"]
archive = result["archive"]

print("\nDatasets (%d):" % len(data))
for k in sorted(data.keys()):
    v = data[k]
    if isinstance(v, np.ndarray):
        print("  %-35s  %s  shape=%s" % (k, v.dtype, v.shape))
    else:
        val = v.decode() if isinstance(v, bytes) else v
        print("  %-35s  = %s" % (k, val))

## 2. Choose Datasets to Export

Set which array datasets to include as CSV columns. Scalar values go into the header.

In [ ]:
########################################
# Which array datasets to export as columns (set to None for all arrays)
COLUMNS = None
# Or pick specific ones:
# COLUMNS = ["mw_scan_x", "mw_retention", "mw_atom_loaded", "mw_atom_retained",
#            "mw_shot1_spcm1", "mw_shot1_spcm2", "mw_shot2_spcm1", "mw_shot2_spcm2",
#            "mw_bg_spcm1", "mw_bg_spcm2", "mw_load_attempts"]
########################################

# Auto-detect: export all arrays with the same length
if COLUMNS is None:
    # Find the most common array length (the per-shot data)
    lengths = {}
    for k, v in data.items():
        if isinstance(v, np.ndarray) and v.ndim == 1 and v.size > 1:
            lengths.setdefault(v.size, []).append(k)
    if lengths:
        main_len = max(lengths, key=lambda l: len(lengths[l]))
        COLUMNS = sorted(lengths[main_len])
        print("Auto-detected %d columns with %d rows:" % (len(COLUMNS), main_len))
    else:
        COLUMNS = []
        print("No array datasets found")
else:
    print("Using %d specified columns" % len(COLUMNS))

for c in COLUMNS:
    print("  %s" % c)

## 3. Export to CSV

In [ ]:
# Output path: same name as H5 but .csv extension
csv_path = filepath.replace(".h5", ".csv")

# Collect scalar metadata for header comment
header_lines = []
header_lines.append("# Source: %s" % os.path.basename(filepath))
for k, v in sorted(meta.items()):
    val = v.decode() if isinstance(v, bytes) else v
    header_lines.append("# %s: %s" % (k, val))

# Add scalar datasets as header comments
for k in sorted(data.keys()):
    v = data[k]
    if not isinstance(v, np.ndarray) or (isinstance(v, np.ndarray) and v.ndim == 0):
        val = v.decode() if isinstance(v, bytes) else v
        if isinstance(v, np.ndarray):
            val = v.item()
        header_lines.append("# %s: %s" % (k, val))

# Build columns
arrays = {}
n_rows = None
for col in COLUMNS:
    if col in data and isinstance(data[col], np.ndarray):
        arr = data[col]
        if n_rows is None:
            n_rows = len(arr)
        if len(arr) == n_rows:
            arrays[col] = arr
        else:
            print("WARNING: %s has %d rows (expected %d), skipping" % (col, len(arr), n_rows))
    else:
        print("WARNING: %s not found or not an array, skipping" % col)

if not arrays:
    print("No data to export!")
else:
    with open(csv_path, 'w', newline='') as f:
        # Write header comments
        for line in header_lines:
            f.write(line + '\n')
        
        # Write CSV
        writer = csv.writer(f)
        cols = sorted(arrays.keys())
        writer.writerow(cols)
        for i in range(n_rows):
            writer.writerow([arrays[c][i] for c in cols])
    
    print("Exported %d rows x %d columns to:" % (n_rows, len(cols)))
    print("  %s" % csv_path)
    print("\nColumns: %s" % ', '.join(cols))

## 4. Export Archive (calibration datasets) to separate CSV

In [ ]:
########################################
EXPORT_ARCHIVE = True  # Set False to skip
########################################

if EXPORT_ARCHIVE and archive:
    archive_csv = filepath.replace(".h5", "_archive.csv")
    with open(archive_csv, 'w', newline='') as f:
        f.write("# Archive (persistent calibration datasets)\n")
        f.write("# Source: %s\n" % os.path.basename(filepath))
        writer = csv.writer(f)
        writer.writerow(["key", "value"])
        for k in sorted(archive.keys()):
            v = archive[k]
            if isinstance(v, bytes):
                v = v.decode()
            elif isinstance(v, np.ndarray):
                v = v.item() if v.ndim == 0 else v.tolist()
            writer.writerow([k, v])
    print("Archive exported (%d keys):" % len(archive))
    print("  %s" % archive_csv)
else:
    print("Archive export skipped.")